# EIS-TIDY3D-LEAN-001 — SOI directional-coupler FDTD

This notebook consumes the JSON emitted by the Lean4 executable and builds a Tidy3D FDTD simulation of a two-waveguide SOI directional coupler. The first parameter set is explicitly illustrative and assumed—not foundry-qualified or measured.

**Cloud execution is opt-in.** The notebook builds and inspects the simulation without submitting it. Set `RUN_CLOUD = True` only after reviewing geometry, data classification, FlexCompute workspace and expected credit use.


In [ ]:
%pip install -r requirements-tidy3d.txt


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
from tidy3d_bridge.directional_coupler import load_spec, build_directional_coupler, launch_cloud

ROOT = Path.cwd()
SPEC_PATH = ROOT / 'examples/tidy3d/eis_soi_dc_spec.json'
spec = load_spec(SPEC_PATH)
print(spec['spec_id'], spec['evidence_class'], spec['process_status'])


## 1. Build the FDTD model

The two guides are separated in the access regions and approach the declared 200 nm gap over the 8 µm parallel coupling section. The lower input mode plane is restricted to the lower arm. Output mode monitors separately record the through and cross ports.


In [ ]:
sim, metadata = build_directional_coupler(spec)
print(metadata)
print('FDTD grid cells:', sim.grid.num_cells)
sim.plot(z=0)
plt.show()


## 2. Explicit cloud-submission gate

No job is submitted unless this cell is deliberately changed. Review FlexCompute account credits and simulation settings before enabling it.


In [ ]:
RUN_CLOUD = False  # Change to True only after explicit review and authorization.
if RUN_CLOUD:
    sim_data = launch_cloud(sim, task_name='EIS-TIDY3D-LEAN-001-SOI-DC', path='data/eis_soi_directional_coupler.hdf5')
else:
    print('Cloud submission skipped. No FlexCompute job has been launched.')


## 3. Inspect modal output (after a completed cloud run)

Keep the raw complex modal amplitudes. Their squared magnitudes are useful modal-power proxies, but the exact normalization convention must be confirmed before comparing absolute power across separate runs. Phase is essential for the EIS complex-field contract.


In [ ]:
if 'sim_data' in globals():
    through = sim_data['through'].amps.sel(direction='+').isel(mode_index=0)
    cross = sim_data['cross'].amps.sel(direction='+').isel(mode_index=0)
    p_through = np.abs(through.values) ** 2
    p_cross = np.abs(cross.values) ** 2
    denom = p_through + p_cross
    cross_fraction = np.divide(p_cross, denom, out=np.full_like(p_cross, np.nan), where=denom > 0)
    wavelengths_um = td.C_0 / through.coords['f'].values
    plt.plot(wavelengths_um, cross_fraction, label='cross-port fraction')
    plt.xlabel('Wavelength (µm)')
    plt.ylabel('Cross-port fraction')
    plt.grid(True)
    plt.legend()
    plt.show()
    print('Retain complex through/cross amplitudes for phase and κA analysis.')
else:
    print('No simulation data available yet.')


## 4. Evidence interpretation and next gate

This simulation can produce a numerical wavelength-dependent complex transfer for the declared geometry. It does **not** establish measured device response, a foundry process model, fabrication tolerance, thermal/electrical drive behavior or KKA system performance.

The next gate is to export complex through/cross amplitudes with the exact Lean spec hash, Python/Tidy3D versions, solver task ID, mesh settings and monitor normalization. Python independently calculates transfer and uncertainty; Lean4 accepts only explicitly stated mathematical predicates. Do not reduce the complex-field contract to amplitude-only power ratios.
